# 04 · Síntese para decisão e recomendações testáveis

**Público:** coordenação, gerência de retenção e cientistas de dados.

**Status:** resultados observacionais da amostra, sujeitos à validação de conceito e confidencialidade. Não publicar métricas reais sem autorização da proprietária dos dados.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
private = root / 'reports' / 'private'

profiles = json.loads((private / 'profiles_summary.json').read_text(encoding='utf-8'))
model = json.loads((private / 'model_validation.json').read_text(encoding='utf-8'))

## Perguntas respondidas

1. Quais configurações de carteira em M0 concentram maior movimentação observada entre retidos?
2. Como a frequência varia por origem e mês?
3. Quais transições econômicas ocorrem no snapshot M0/M30?
4. O evento composto pode ser previsto fora do tempo melhor que a baseline?

In [ ]:
mix = [g for g in profiles['segments'] if g['dimension'] == 'family_count_m0' and g['n_retained'] >= 100]
for group in sorted(mix, key=lambda g: float(g['category'])):
    print(group['category'], group['n_retained'],
          round(100 * group['movement_among_retained'], 2))
print('Teste:', model['test_metrics'])

## Recomendações formuladas como hipóteses

| Prioridade | Hipótese a testar | Público candidato | Indicador primário | Salvaguarda |
|---|---|---|---|---|
| 1 | Melhorar a aderência de ofertas ao mix de produtos do início do atendimento | Estratos por quantidade e combinação de produtos em M0 | Retenção operacional entre leads encerrados, por coorte e canal | Acompanhar valor M0/M30 separadamente |
| 2 | Investigar diferenças operacionais por origem e mês | Canais SOLAR/LEGADO e equipes | Distribuição de encerrados/abertos e movimentação por status | Controlar mix e maturidade; não atribuir causalidade ao canal |
| 3 | Revisar casos com variação negativa de valor após retenção | Retidos com movimentação e delta negativo | Distribuição de delta de valor e posterior margem/receita validada | Não rotular desconto como perda de receita realizada sem contrato contábil |
| 4 | Avaliar sinalização preditiva em shadow mode | Novos leads com atributos M0 validados | Brier, AP, lift@10%, estabilidade temporal | Nenhuma intervenção baseada no escore sem experimento aprovado |

**Desenho recomendado:** testes randomizados por lead/cliente após estabelecer chave e guardrails, com pré-registro de métricas de retenção, receita e reclamações. Nenhum impacto financeiro incremental é estimado aqui.

## Critérios ainda abertos

A chave oficial do episódio e a elegibilidade; o significado de `RETIDO`, `FLAG_MOV_*` e `VALOR_*`; a janela efetiva de desfecho; e autorização de divulgação dos agregados. Registrar responsáveis e decisões em `docs/decision_log.md`. A modelagem exploratória **não** encerra essas pendências.